# Конспект: Обработка ошибок, создание исключений и логирование в Python

### Дополнение: сначала - как выглядит ошибка

Ошибки Вы уже встречали в прошлых темах. Когда что-то идёт не так, Python
останавливает программу и печатает сообщение. Его стоит читать **снизу вверх**:

- **последняя строка** - тип ошибки и её текст: `IndexError: list index out of range`;
- **стрелка `---->`** - строка кода, на которой всё сломалось;
- выше - путь, которым программа пришла к этой строке (если ошибка случилась внутри функции).

In [ ]:
titles = ['Сборная вышла в четвертьфинал', 'Тренер покинул клуб']

print('первый:', titles[0])
print('пятый:', titles[4])     # ячейка специально заканчивается ошибкой
print('эта строка уже не выполнится')

первый: Сборная вышла в четвертьфинал


IndexError: list index out of range

Такую ошибку Python называет **исключением** (exception): ситуация «исключительная»,
продолжать по плану нельзя. Частые исключения, которые вы уже видели:

| Исключение | Когда возникает | Пример |
| --- | --- | --- |
| `ValueError` | значение неподходящее | `int('три')` |
| `TypeError` | тип неподходящий | `'5' + 3`, `int(None)` |
| `ZeroDivisionError` | деление на ноль | `10 / 0` |
| `IndexError` | нет такого индекса | `[1, 2][5]` |
| `KeyError` | нет такого ключа | `{'a': 1}['b']` |
| `NameError` | имя не определено | опечатка в имени переменной |
| `AttributeError` | у объекта нет такого метода | `None.split()` |
| `FileNotFoundError` | файла нет | `open('nope.txt')` |

Конструкция `try-except` позволяет **перехватить** исключение и решить, что делать
дальше, вместо того чтобы программа просто останавливала свою работу.

## 1. Конструкция try-except
### Что такое try-except?

Конструкция `try-except` в Python используется для обработки исключений, т.е. ошибок, которые возникают во время выполнения программы. Вместо того, чтобы программа завершилась аварийно, с помощью `try-except` можно перехватить ошибку и выполнить альтернативное действие.
- Синтаксис:
```python
try:
    # Код, который может вызвать ошибку
except ExceptionType:
    # Код, который выполняется при возникновении ошибки
```
Ниже представлена визуализация всех этапов работы `try-except`:  
![image.png](attachment:cbacd306-5a20-475d-ad85-4becb3f910a3.png)

In [ ]:
try:
    x = 10 / 0
except ZeroDivisionError:
    print("Ошибка: Деление на ноль!")

Ошибка: Деление на ноль!


## Как это работает:

В блоке try программа пытается выполнить потенциально опасный код. Если возникает ошибка (в данном случае ZeroDivisionError), выполнение переходит в блок except.
Блок except перехватывает указанную ошибку и выполняет соответствующий код.

### **Особенности**:

Можно обрабатывать несколько типов ошибок с помощью нескольких блоков except.
Можно перехватить все ошибки с помощью общего исключения Exception, но это плохая практика, так как не всегда ясно, какая именно ошибка произошла.
Пример с несколькими исключениями:
```python
try:
    x = int(input("Введите число: "))
    y = 10 / x
except ValueError:
    print("Ошибка: Введено не число.")
except ZeroDivisionError:
    print("Ошибка: Деление на ноль.")
```

В примере выше используется `input()`. Чтобы увидеть все ветки сразу, возьмём
несколько «ответов пользователя» и проверим каждый в цикле. Python проверяет блоки
`except` **сверху вниз** и выполняет первый подходящий.

In [ ]:
for answer in ['5', '0', 'пять']:
    try:
        x = int(answer)
        y = 10 / x
        print(answer, '->', y)
    except ValueError:
        print(answer, '-> Ошибка: Введено не число.')
    except ZeroDivisionError:
        print(answer, '-> Ошибка: Деление на ноль.')

5 -> 2.0
0 -> Ошибка: Деление на ноль.
пять -> Ошибка: Введено не число.


### `except ... as error` - текст ошибки

Чтобы узнать, что именно случилось, исключение сохраняют в переменную: `as error`.
`print(error)` печатает текст ошибки, `type(error).__name__` её тип. Несколько типов
можно перехватить одним блоком, записав их в скобках.

In [ ]:
for value in ['42', 'сорок два', None]:
    try:
        print(int(value) + 1)
    except (ValueError, TypeError) as error:
        print(type(error).__name__, '-', error)

43
ValueError - invalid literal for int() with base 10: 'сорок два'
TypeError - int() argument must be a string, a bytes-like object or a real number, not 'NoneType'


### `else` - если ошибки не было

У `try` есть ещё блок `else`, который выполняется, только если в `try` ошибки **не было**.
В `try` оставляют только опасную строку, а то, что зависит от её успеха, пишут в `else`.

| Что случилось в `try` | Что выполнится дальше |
| --- | --- |
| ошибки нет | `else`, потом `finally` |
| ошибка и есть подходящий `except` | этот `except`, потом `finally` |
| ошибка и подходящего `except` нет | `finally`, потом программа остановится с ошибкой |

In [ ]:
for answer in ['4', 'четыре']:
    try:
        number = int(answer)
    except ValueError as error:
        print('ошибка:', error)
    else:
        print('удвоенное:', number * 2)

удвоенное: 8
ошибка: invalid literal for int() with base 10: 'четыре'


### Неподходящий `except` ошибку не спасает

`except ValueError` перехватывает **только** `ValueError`. Если случилась другая ошибка,
программа остановится, как будто `try` не было.

In [ ]:
counts = {'спорт': 2, 'наука': 3}

try:
    print(counts['культура'])
except ValueError:                 # ждём не ту ошибку
    print('не число')
# ячейка специально заканчивается ошибкой KeyError

KeyError: 'культура'

**`Exception` перехватывает почти всё.** Поэтому, во-первых, его ставят **последним**,
блоки проверяются сверху вниз, и после `except Exception` до более точных дело не дойдёт.
Во-вторых, не стоит писать `except Exception: pass` или просто `except: pass`, такой код
пропустит любые ошибки, включая опечатки и потом очень трудно понять, почему
результат неверный.

In [ ]:
try:
    print(int('пять'))
except Exception as error:         # стоит первым - перехватит и ValueError
    print('что-то пошло не так:', error)
except ValueError:
    print('этот блок никогда не выполнится')

что-то пошло не так: invalid literal for int() with base 10: 'пять'


## 2. Использование finally для завершения операций

**Что такое finally**?

Блок `finally` используется для выполнения кода, который должен быть выполнен независимо от того, возникло исключение или нет. Обычно это освобождение ресурсов, закрытие файлов или завершение соединений с базой данных.

**Синтаксис**:

```python
try:
    # Код, который может вызвать ошибку
except ExceptionType:
    # Код, который выполняется при возникновении ошибки
finally:
    # Код, который выполняется в любом случае
```

### Пример использования:

In [ ]:
# файла с таким именем нет - так ошибка видна на любом компьютере
# ячейка специально заканчивается ошибкой NameError
try:
    file = open('no_such_file.txt', 'r', encoding='utf-8')
    content = file.read()
except FileNotFoundError:
    print("Файл не найден!")
finally:
    print("Закрываем файл.")
    file.close()

Файл не найден!
Закрываем файл.


NameError: name 'file' is not defined

Программа вызывает ошибку `NameError`. Файла нет, поэтому `open()` упал с `FileNotFoundError` и не вернул никакого файла - присваивание `file = ...` просто не выполнилось. Блок `except` обработал ошибку, а затем `finally` попытался закрыть файл, которого не существует.

> Если в ноутбуке раньше уже была переменная `file` (например, из другой ячейки), ошибки не будет: `finally` молча закроет **тот, старый** файл. Такие ошибки самые коварные.

Чтобы избежать таких ошибок, лучше инициализировать переменную `file` перед блоком `try-except` и использовать более безопасный подход с контекстным менеджером `with`, который автоматически закроет файл после его использования. Это устранит необходимость явно закрывать файл в блоке `finally`.


### Вот несколько решений:

## 1. Инициализация переменной перед `try-except`:

In [ ]:
file = None  # Инициализация переменной file

try:
    file = open('example.txt', 'r', encoding='utf-8')
    content = file.read()
except FileNotFoundError:
    print("Файл не найден!")
finally:
    print("Закрываем файл.")
    if file is not None:
        file.close()

Файл не найден!
Закрываем файл.


Переменная `file` инициализируется как `None` до блока `try`.
Если файл не найден, блок `except` выполнится, но переменная `file` останется равной `None`, и проверка в блоке `finally` предотвратит попытку закрытия несуществующего файла.

## 2. Использование контекстного менеджера with:

In [ ]:
try:
    with open('example.txt', 'r', encoding='utf-8') as file:
        content = file.read()
        print(content)
except FileNotFoundError:
    print("Файл не найден!")
finally:
    print("Операция завершена.")

Файл не найден!
Операция завершена.


Контекстный менеджер with автоматически закрывает файл после выхода из блока, независимо от того, произошла ошибка или нет.
Здесь нет необходимости явно закрывать файл в блоке finally, что делает код проще и безопаснее.

Таким образом, контекстный менеджер with, является лучшим подходом для работы с файлами, так как он автоматически управляет открытием и закрытием файлов, избегая ошибок и облегчая обработку ресурсов.

Для двух вариантов ниже нужен файл `example.txt`. Создадим его, как в теме 4:

In [ ]:
with open('example.txt', 'w', encoding='utf-8') as file:
    file.write('Привет мир!\n')
    file.write('Hello World!\n')
    file.write('Третья строка.\n')

In [ ]:
# Вариант №1 Инициализация переменной перед try-except:
file = None  # Инициализация переменной file

try:
    file = open('example.txt', 'r', encoding='utf-8')
    content = file.read()
except FileNotFoundError:
    print("Файл не найден!")
finally:
    print("Закрываем файл.")
    if file is not None:
        file.close()

Закрываем файл.


In [ ]:
# Вариант №2 Использование контекстного менеджера with:

try:
    with open('example.txt', 'r', encoding='utf-8') as file:
        content = file.read()
        print(content)
except FileNotFoundError:
    print("Файл не найден!")
finally:
    print("Операция завершена.")

Привет мир!
Hello World!
Третья строка.

Операция завершена.


В результате, независимо от того, будет ли ошибка в блоке `try`, блок `finally` выполнится всегда. Это гарантирует выполнение операций, необходимых для завершения работы.
*Примечание*: Даже если в блоке `try` возникло исключение, блок `finally` выполнится до выхода из программы.

### Ошибки в проверке данных при помощи цикла

В реальных данных всегда есть неподходящие для обработки -
пустые строки, опечатки, `None`. Одна плохая запись не должна ломать обработку всех
остальных, обычно её пропускают, считают и идут дальше. Для этого `try` используют **внутри** цикла.

In [ ]:
raw_scores = ['5', '3', '', 'пять', '4', None]

scores = []
skipped = 0
for value in raw_scores:
    try:
        scores.append(int(value))
    except (ValueError, TypeError):
        skipped += 1

print('оценки:', scores)
print('пропущено:', skipped)
print('средняя:', sum(scores) / len(scores))

оценки: [5, 3, 4]
пропущено: 3
средняя: 4.0


Если использовать `try` **снаружи** цикла, первая же плохая запись прервёт весь цикл,
и остальные хорошие записи не будут обработаны.

### `raise` или сообщить об ошибке самому

Ошибку можно вызвать и самому командой `raise`. Так функция сообщает: «с такими данными
я работать не буду». Для этого подходят встроенные исключения,
`raise ValueError('текст')`.

In [ ]:
def word_count(text):
    """Число слов в тексте. Пустой текст - ошибка."""
    if text.strip() == '':
        raise ValueError('пустой текст')
    return len(text.split())

print(word_count('Сборная выиграла матч'))
print(word_count('   '))           # ячейка специально заканчивается ошибкой

3


ValueError: пустой текст

Посмотрите на сообщение: ошибка возникла внутри `word_count`, а сообщение показывает
обе строки - где функцию **вызвали** и где внутри неё случился `raise`. Ошибка «всплывает»
из функции туда, откуда её вызвали. Там её и перехватывают:

In [ ]:
for text in ['Сборная выиграла матч', '   ', 'Ставка снижена']:
    try:
        print(word_count(text), 'слова')
    except ValueError as error:
        print('ошибка:', error)

3 слова
ошибка: пустой текст
2 слова


### `if` или `try`?

Часть ошибок проще не допустить, чем перехватывать:

| Ситуация | Через проверку | Через `try` |
| --- | --- | --- |
| ключа может не быть | `counts.get(key, 0)` или `if key in counts:` | `except KeyError` |
| строка может быть не числом | `if value.isdigit():` (только целые без знака) | `except ValueError` |
| файла может не быть | - | `except FileNotFoundError` |

Если проверка простая и понятная - пишите `if`. Если заранее проверить трудно
(`'3.5'`, `'-2'`, `' 7 '` - всё это числа для `float`/`int`, но не для `isdigit`) - `try`.

In [ ]:
counts = {'спорт': 2, 'наука': 3}

print(counts.get('культура', 0))           # без ошибки

for value in ['7', '-2', ' 7 ']:
    print(repr(value), 'isdigit:', value.isdigit(), '| int:', int(value))

0
'7' isdigit: True | int: 7
'-2' isdigit: False | int: -2
' 7 ' isdigit: False | int: 7


# Забегая немного вперед
## 3. Создание и использование собственных исключений
### Зачем создавать свои исключения?

Иногда стандартных исключений недостаточно для описания специфических ошибок вашей программы. В таких случаях можно создать свои собственные исключения, которые помогут лучше обрабатывать и отлавливать ошибки в приложении.

### Как создать собственное исключение?

Для создания собственного исключения в Python нужно создать класс, который наследуется от встроенного класса Exception.

Пример:

> Классы мы будем проходить позже в курсе. Сейчас достаточно понимать, что своё исключение -
> это **новое имя** для ошибки. Строки `class CustomError(Exception): pass` хватает,
> чтобы потом писать `raise CustomError(...)` и `except CustomError`. Для большинства задач
> подходят и встроенные `ValueError`, `TypeError`, как в `raise` выше.

In [ ]:
class CustomError(Exception):
    pass

def divide(x, y):
    if y == 0:
        raise CustomError("Деление на ноль запрещено.")
    return x / y

try:
    result = divide(10, 0)
except CustomError as e:
    print(f"Ошибка: {e}")

Ошибка: Деление на ноль запрещено.


### Объяснение:

Здесь мы создали собственное исключение `CustomError`, которое вызывается в случае попытки деления на ноль.
Затем в блоке `try-except` мы ловим это исключение и выводим сообщение об ошибке.

Вы также можете добавлять атрибуты к собственным исключениям для передачи дополнительных данных (более подробно об этом узнаем попозже в дальгейших лекциях):

In [ ]:
class CustomError(Exception):
    def __init__(self, message, value):
        self.message = message
        self.value = value
        super().__init__(message)

try:
    raise CustomError("Неверное значение", 42)
except CustomError as e:
    print(f"Ошибка: {e.message}, значение: {e.value}")

Ошибка: Неверное значение, значение: 42


### Журнал ошибок своими руками

Прежде чем подключать модуль `logging`, посмотрим на идею лога без него. Лог - это обычный
текстовый файл, в конец которого программа дописывает строки о том, что произошло.
Режим `'a'` из темы 4 для этого и нужен.

In [ ]:
with open('my_log.txt', 'w', encoding='utf-8') as log:          # начинаем журнал заново
    log.write('INFO: старт обработки\n')

for value in ['5', 'пять', '4']:
    try:
        score = int(value)
    except ValueError as error:
        with open('my_log.txt', 'a', encoding='utf-8') as log:
            log.write(f'ERROR: {error}\n')
    else:
        with open('my_log.txt', 'a', encoding='utf-8') as log:
            log.write(f'INFO: принята оценка {score}\n')

with open('my_log.txt', 'r', encoding='utf-8') as log:
    print(log.read())

INFO: старт обработки
INFO: принята оценка 5
ERROR: invalid literal for int() with base 10: 'пять'
INFO: принята оценка 4



Модуль `logging` делает то же самое, но умеет больше: время события, уровни важности,
фильтр по уровню, запись одновременно на экран и в файл. `import` подробно будет рассмотрен в теме 6.

## Заключение

- **try-except** позволяет обрабатывать ошибки, предотвращая аварийное завершение программы.
- **finally** гарантирует выполнение важного кода, даже если возникла ошибка.
- **Собственные исключения** помогают описывать специфические ошибки программы, делая её более понятной и управляемой.

---

## Шпаргалка

| Что нужно | Как пишется | Важно |
| --- | --- | --- |
| перехватить ошибку | `try: ...` `except ValueError: ...` | ловите ожидаемый тип |
| несколько типов | `except ValueError: ...` `except KeyError: ...` | проверяются сверху вниз |
| несколько типов одним блоком | `except (ValueError, TypeError):` | |
| текст ошибки | `except ValueError as error: print(error)` | тип: `type(error).__name__` |
| если ошибки не было | `else:` | в `try` - только опасная строка |
| выполнить в любом случае | `finally:` | закрыть файл, напечатать итог |
| пропустить плохую запись | `try` **внутри** цикла | считайте пропущенные |
| сообщить об ошибке | `raise ValueError('текст')` | ошибка всплывает к месту вызова |
| нет файла | `except FileNotFoundError:` | |
| без ошибки по ключу | `d.get(key, 0)` | вместо `except KeyError` |
| всё подряд | `except Exception as error:` | последним и не с `pass` |
| свой тип ошибки | `class MyError(Exception): pass` | классы - позже в курсе |
| лог в файл | `logging.basicConfig(filename=..., level=..., encoding='utf-8', force=True)` | |

---

## Найди ошибку

Раскомментируйте, запустите и найдите причину. Ответы - после ячеек.

In [ ]:
# Ошибка 1
# values = [10, 5, 0]
# try:
#     print(100 / values[2])
# except ValueError:
#     print('плохое значение')

In [ ]:
# Ошибка 2 - ошибки нет, но учтены только две оценки из четырёх
# raw = ['5', '4', 'пять', '3']
# scores = []
# try:
#     for value in raw:
#         scores.append(int(value))
# except ValueError:
#     pass
# print(scores)

In [ ]:
# Ошибка 3 - ошибки нет, но сообщение не то
# try:
#     print(int('пять'))
# except Exception:
#     print('неизвестная ошибка')
# except ValueError:
#     print('это не число')

In [ ]:
# Ошибка 4 - ошибки нет, но проверка не сработала
# def mean(values):
#     if len(values) == 0:
#         ValueError('пустой список')
#     return sum(values) / len(values)
#
# print(mean([]))

In [ ]:
# Ошибка 5
# try:
#     n = int('пять')
# except ValueError:
#     print('не число')
#
# print(n * 2)

**Ответы:**

1. `ZeroDivisionError: division by zero`. Деление на ноль - не `ValueError`, поэтому
   `except` его не перехватил. Нужно `except ZeroDivisionError`.
2. `try` стоит снаружи цикла: на `'пять'` цикл прервался, и `'3'` уже не обработана.
   А `pass` скрыл, что что-то пошло не так. Нужен `try` внутри цикла и счётчик пропущенных.
3. `except Exception` стоит первым и перехватывает и `ValueError`. Точный блок нужно
   поставить выше общего.
4. `ValueError('пустой список')` только создаёт исключение, но не вызывает его - забыто
   слово `raise`. В итоге `ZeroDivisionError` в следующей строке.
5. `NameError: name 'n' is not defined`. `int('пять')` упал, и присваивание не выполнилось:
   переменной `n` нет. То, что зависит от успеха `try`, пишут в `else`.

---

# Подробнее о логировании

> **Необязательное чтение.** Обзорная статья для тех, кто хочет узнать, как логирование
> устроено в больших проектах. Примеры в ней - для отдельных `.py`-программ, а не для
> Jupyter, и используют библиотеки, которые нужно устанавливать. Для практики №5 она
> не нужна.

## Инструменты логирования (модули logging и loguru)
### Модуль logging
### Зачем использовать логирование?

**Логирование** - это процесс записи информации о работе программы. Логи помогают отслеживать, что происходило в программе, выявлять ошибки и анализировать поведение системы.

Модуль `logging` - это встроенный модуль в Python, который позволяет записывать сообщения разного уровня важности (отладка, информация, предупреждение, ошибка, критическая ошибка).

### Основные уровни логирования:

- **DEBUG**: Подробная информация, полезная для отладки.
- **INFO**: Информационные сообщения, описывающие работу программы.
- **WARNING**: Предупреждения о потенциальных проблемах.
- **ERROR**: Ошибки, которые не привели к аварийному завершению программы.
- **CRITICAL**: Критические ошибки, которые привели к сбою программы.

### Пример использования logging:

In [ ]:
import logging

# Настройка логирования
logging.basicConfig(level=logging.DEBUG, filename='app.log', filemode='w',
                    format='%(levelname)s: %(message)s', encoding='utf-8', force=True)

logging.debug('Это сообщение для отладки')
logging.info('Это информационное сообщение')
logging.warning('Это предупреждение')
logging.error('Произошла ошибка')
logging.critical('Критическая ошибка!')

### Дополнение: что попало в `app.log`

На экран `logging` здесь ничего не выводит, т.к. всё записалось в файл. Откроем его.

In [ ]:
with open('app.log', 'r', encoding='utf-8') as file:
    print(file.read())

DEBUG: Это сообщение для отладки
INFO: Это информационное сообщение
ERROR: Произошла ошибка
CRITICAL: Критическая ошибка!



Два параметра в `basicConfig` добавлены к исходному примеру:

- `encoding='utf-8'` - чтобы русские сообщения не превратились в непонятные символы;
- `force=True` - **важно в Jupyter**. `basicConfig` настраивает логирование только
  **один раз**, а повторные вызовы в том же ядре Python пропускает. Без `force=True`
  второй `basicConfig` (с другим файлом или уровнем) просто не сработает. В примерах
  решений ниже это как раз случилось, в задачах 4-6 логи записывались не в те файлы.

### Порядок уровня отсекает сообщения

`level` - минимальная важность. Сообщения ниже этого уровня не записываются.
Порядок: `DEBUG < INFO < WARNING < ERROR < CRITICAL`.

In [ ]:
logging.basicConfig(level=logging.WARNING, filename='app.log', filemode='w',
                    format='%(levelname)s: %(message)s', encoding='utf-8', force=True)

logging.debug('не попадёт')
logging.info('не попадёт')
logging.warning('попадёт')
logging.error('попадёт')

with open('app.log', 'r', encoding='utf-8') as file:
    print(file.read())

ERROR: попадёт



`basicConfig` настраивает параметры логирования: уровень, файл для записи логов, режим работы файла (`w` — перезапись, `a` — добавление).
Сообщения записываются в файл `app.log`.

## Модуль `loguru`
### Что такое loguru?

**loguru** — это сторонний модуль для логирования, который упрощает процесс настройки и использования логов по сравнению с `logging`. Он предоставляет более простой интерфейс и функциональные возможности "из коробки", такие как цветные логи в консоли и ротация файлов.

### Установка:

`loguru` - сторонняя библиотека, её нужно установить. В Jupyter удобнее команда `%pip`:
она ставит пакет именно туда, где работает ваша тетрадь. Вариант `!python.exe -m pip ...`
из исходной версии работает только в Windows.

In [ ]:
%pip install loguru

## Пример использования loguru:

In [ ]:
from loguru import logger

# Запись логов в файл с ротацией (создание новых файлов при достижении определенного размера)
logger.add("app.log", rotation="500 KB")

logger.debug("Это сообщение для отладки")
logger.info("Это информационное сообщение")
logger.warning("Это предупреждение")
logger.error("Произошла ошибка")
logger.critical("Критическая ошибка!")

> Вывод этой ячейки не сохранён: при подготовке файла `loguru` не была установлена.
> Запустите ячейку у себя после установки.
>
> **Осторожно в Jupyter:** каждый запуск `logger.add("app.log")` **добавляет ещё один**
> вывод в файл, и после трёх запусков каждая строка будет записываться трижды. Если
> ячейку нужно перезапускать, начните её с `logger.remove()`. И лучше писать в отдельный
> файл (`loguru.log`), а не в тот же `app.log`, что и `logging`.

### Объяснение:

`loguru` автоматически выводит логи в консоль и записывает их в файл.
Ротация файлов означает, что новые файлы создаются при достижении определенного размера (в данном примере 500 KB).
Основные преимущества `loguru`:

- Простота настройки: не нужно конфигурировать формат логов и уровни, как в `logging`.
- Поддержка ротации файлов и отправки логов по сети.
- Цветные сообщения в консоли для лучшей читаемости.

# Примеры решений задач

> **Изменения в этой версии.** Во все `open()` добавлен `encoding='utf-8'`. Во все
> `logging.basicConfig` добавлены `encoding='utf-8'` и `force=True`: без него в одной
> тетради сработает только первая настройка, и логи задач 4-6 окажутся в `app.log`
> из задачи 3 с уровнем ERROR (сообщения INFO из задач 5 и 6 не запишутся вовсе).
> Задачи используют `input()`, поэтому оставлены текстом - скопируйте код в ячейку,
> чтобы попробовать.

## Задача 1: Обработка деления на ноль и ошибки ввода
Задача: Напишите программу, которая запрашивает у пользователя два числа и делит первое на второе. Если пользователь вводит не число или пытается разделить на ноль, программа должна вывести соответствующее сообщение об ошибке, но не завершиться аварийно.

### Решение:

```python
try:
    # Вводим первое число
    num1 = float(input("Введите первое число: "))
    # Вводим второе число
    num2 = float(input("Введите второе число: "))
    # Выполняем деление
    result = num1 / num2
except ValueError:
    print("Ошибка: Введено не число.")
except ZeroDivisionError:
    print("Ошибка: Деление на ноль невозможно.")
else:
    print(f"Результат деления: {result}")
finally:
    print("Завершение программы.")
```

## Задача 2: Создание собственного исключения
Задача: Создайте собственное исключение InvalidAgeError. Если пользователь вводит возраст меньше 0 или больше 120, выбросьте это исключение и обработайте его.

### Решение:

```python
# Создание собственного исключения
class InvalidAgeError(Exception):
    def __init__(self, age):
        self.age = age
        super().__init__(f"Некорректный возраст: {age}")

def get_age():
    try:
        age = int(input("Введите ваш возраст: "))
        if age < 0 or age > 120:
            raise InvalidAgeError(age)
        print(f"Ваш возраст: {age}")
    except InvalidAgeError as e:
        print(f"Ошибка: {e}")
    except ValueError:
        print("Ошибка: Введено не число.")

get_age()
```
## Задача 3: Логирование ошибок при работе с файлами
Задача: Программа пытается открыть текстовый файл для чтения. Если файл не существует, программа записывает сообщение об ошибке в лог-файл.

### Решение:

```python
import logging

# Настройка логирования
logging.basicConfig(filename='app.log', level=logging.ERROR, format='%(asctime)s %(levelname)s:%(message)s',
                    encoding='utf-8', force=True)

def read_file(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            content = file.read()
            print(content)
    except FileNotFoundError:
        logging.error(f"Файл {filename} не найден!")
        print(f"Ошибка: Файл {filename} не найден!")

# Вызов функции
read_file('example.txt')
```

## Задача 4: Обработка ошибок и логирование в функции
Задача: Напишите функцию, которая принимает строку с именем файла, пытается открыть его и считать данные. Если файл не найден или произошла ошибка при чтении, запишите соответствующее сообщение в лог-файл с уровнем ERROR.

### Решение:

```python
import logging

# Настройка логирования
logging.basicConfig(filename='file_operations.log', level=logging.ERROR, format='%(asctime)s %(levelname)s:%(message)s',
                    encoding='utf-8', force=True)

def read_file(filename):
    file = None
    try:
        file = open(filename, 'r', encoding='utf-8')
        content = file.read()
        print(content)
    except FileNotFoundError:
        logging.error(f"Файл {filename} не найден!")
        print(f"Ошибка: Файл {filename} не найден!")
    except IOError:
        logging.error(f"Ошибка чтения файла {filename}")
        print(f"Ошибка: Невозможно прочитать файл {filename}")
    finally:
        if file is not None:
            file.close()

# Вызов функции
read_file('missing_file.txt')
```

## Задача 5: Логирование успешного и неуспешного выполнения программы
Задача: Напишите программу, которая считывает два числа, выполняет деление и выводит результат. Добавьте логирование как ошибок, так и успешных операций с указанием времени выполнения программы.

### Решение:

```python
import logging

# Настройка логирования
logging.basicConfig(filename='division.log', level=logging.INFO, format='%(asctime)s %(levelname)s:%(message)s',
                    encoding='utf-8', force=True)

def divide_numbers():
    try:
        num1 = float(input("Введите первое число: "))
        num2 = float(input("Введите второе число: "))
        result = num1 / num2
        logging.info(f"Успешное деление: {num1} / {num2} = {result}")
        print(f"Результат деления: {result}")
    except ZeroDivisionError:
        logging.error("Ошибка: Деление на ноль невозможно.")
        print("Ошибка: Деление на ноль невозможно.")
    except ValueError:
        logging.error("Ошибка: Введено не число.")
        print("Ошибка: Введено не число.")

# Вызов функции
divide_numbers()
```
## Задача 6: Проверка правильности ввода с логированием
Задача: Программа запрашивает у пользователя число, проверяет, является ли оно четным, и записывает результат в лог-файл.

### Решение:

```python
import logging

# Настройка логирования
logging.basicConfig(filename='even_check.log', level=logging.INFO, format='%(asctime)s %(levelname)s:%(message)s',
                    encoding='utf-8', force=True)

def check_even():
    try:
        number = int(input("Введите число: "))
        if number % 2 == 0:
            logging.info(f"Пользователь ввел четное число: {number}")
            print(f"Число {number} четное.")
        else:
            logging.info(f"Пользователь ввел нечетное число: {number}")
            print(f"Число {number} нечетное.")
    except ValueError:
        logging.error("Ошибка: Введено не число.")
        print("Ошибка: Введено не число.")

# вызов функции
check_even()
```
## Задача 7: Использование loguru для логирования
Задача: Программа запрашивает у пользователя путь к файлу и пытается открыть его. Если файл не найден, программа записывает сообщение об ошибке в лог с использованием библиотеки loguru.

### Решение:

```python
from loguru import logger

# Настройка loguru для записи в файл
logger.add("file_logs.log", format="{time} {level} {message}", level="ERROR", encoding="utf-8")

def read_file(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as file:
            content = file.read()
            print(content)
    except FileNotFoundError:
        logger.error(f"Файл {filename} не найден!")
        print(f"Ошибка: Файл {filename} не найден!")

# Пример вызова функции
read_file('nonexistent.txt')
```